# Project 5: LoRA supervised fine-tuning for candle-data repair instructions

**Name:** To be completed  
**Course:** CSCI 4820 — section to be completed

`candleprep` downloads the original CSV from the public NLP-proj5 repository,
creates corruptions, and verifies their JSON repair labels offline. This notebook
displays the data and adapts the starter's Mistral/PEFT `Trainer` workflow.
The model receives only corrupted CSV and predicts a repair plan, never the
original values, corruption history, or reconstructed data.

Use a CUDA-enabled PyTorch environment with Transformers installed. The setup
cell installs PEFT 0.21.3, bitsandbytes 0.50.2, and Accelerate 1.15.0 alongside
candleprep. These versions were verified locally with Transformers 5.8.1.
After the first installation, restart the kernel and run from the top.
Run from top to bottom. The first source/model downloads need internet access;
later runs reuse local caches. Full training and evaluation can take hours.

## 1. Install the CPU preparation package

Install the data/display package and the LoRA/4-bit training dependencies in
the active kernel. No separate CSV download or repository clone is required.
After installation, restart the kernel once before continuing.

Math rendering is disabled for styled data tables so currency signs remain
literal text. This changes presentation only, not model inputs or labels.


In [ ]:
%pip install -q "candleprep[notebook]==0.2.0" "peft==0.21.3" "accelerate==1.15.0" "bitsandbytes==0.50.2"
from candleprep import CandleDataset
import pandas as pd
pd.set_option("styler.html.mathjax", False)  # Display currency signs as literal text


## 2. Prepare the dataset

One call downloads the pinned original CSV, verifies its checksum, and creates
10,000 training, 1,000 validation, and 2,000 test pairs. Existing matching files
are verified and reused. Source windows are split before corruption to avoid
leakage. All labels are produced by deterministic tools, without model inference.


In [ ]:
dataset = CandleDataset.prepare(output_dir="generated_data/v1", seed=4820)
dataset.summary()


## 3. Inspect the data and the actual SFT labels

Display the first, middle, and last five rows of the original and synthetic data.
The label table shows three recoverable examples and two whole-window omissions.
Its reconstruction is evidence that the JSON label is correct. The supervised
target remains the JSON repair instructions, not reconstructed CSV.


In [ ]:
dataset.display_samples(rows=5)
dataset.display_labels(rows=5)
dataset.display_coverage()


## 4. Model imports and experiment settings

Retain the starter's 4-bit Mistral and LoRA approach. Batch size 1, accumulation
16, and gradient checkpointing preserve the tested low-memory configuration.
The previous rank-16 run peaked at 5.90 GiB allocated / 6.17 GiB reserved by
PyTorch; whole-device peak was not logged, so a 10-GB GPU still needs verification.

After an interruption stops, call `free_gpu_resources()` in a new cell. It clears
live model references and traceback/output caches; saved checkpoints remain.
After a kernel restart, rerun setup/definition cells before running later cells.
Saved outputs alone do not restore Python variables.


In [ ]:
import gc, hashlib, json, math, time
from pathlib import Path
import pandas as pd
import torch
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from candleprep.task import TASK, user_text
from candleprep.evaluation import score_predictions

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
RESULTS = Path("candle_results/simple_sft_v2")
SEED = 4820
LORA_RANK = 16  # Primary adapter used for the before/after comparison
RANKS = [8, 16, 32]  # Separate rank study required by the assignment
EPOCHS = 1
BATCH_SIZE = 1
ACCUMULATION = 16
LEARNING_RATE = 0.0002
MAX_LENGTH = 1536
MAX_NEW_TOKENS = 128
RUN_RANK_STUDY = True
RUN_FULL_TEST_COMPARISON = True
RUN_FINAL_DEMO = False
RESULTS.mkdir(parents=True, exist_ok=True)

def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")

settings = {"model": MODEL_ID, "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
            "accumulation": ACCUMULATION, "learning_rate": LEARNING_RATE,
            "max_length": MAX_LENGTH, "max_new_tokens": MAX_NEW_TOKENS,
            "task_sha256": hashlib.sha256(TASK.encode()).hexdigest(),
            "dataset_artifacts": dataset.manifest["artifact_sha256"]}
configuration = RESULTS / "settings.json"
if configuration.exists() and json.loads(configuration.read_text()) != settings:
    raise ValueError("Run settings changed; choose a new RESULTS directory")
save_json(configuration, settings)
print("Optional jobs:", {"rank_study": RUN_RANK_STUDY, "full_test": RUN_FULL_TEST_COMPARISON})


### Manual GPU cleanup

Run this definition cell once. After an interrupted run stops, call
`free_gpu_resources()` in a new cell; saved checkpoints remain available.


In [ ]:
def free_gpu_resources():
    """Call in a new cell after an interrupted run has stopped."""
    import sys
    import traceback
    import torch

    # Drop this notebook's live models/training objects; keep data and checkpoints.
    for name in ("trainer", "model", "base_model", "optimizer", "scheduler"):
        globals().pop(name, None)

    # An interrupted Trainer/generate call can survive through traceback locals.
    for name in ("last_traceback", "last_value", "last_exc"):
        value = getattr(sys, name, None)
        tb = value if name == "last_traceback" else getattr(value, "__traceback__", None)
        if tb is not None:
            traceback.clear_frames(tb)
        if hasattr(sys, name):
            setattr(sys, name, None)
    shell = globals().get("get_ipython", lambda: None)()
    if shell is not None:
        shell.run_line_magic("reset", "-f out")  # Clear cached Python results.
        shell.last_execution_result = None
        tb = getattr(shell.InteractiveTB, "tb", None)
        if tb is not None:
            traceback.clear_frames(tb)
            shell.InteractiveTB.tb = None
    gc.collect()
    if not torch.cuda.is_initialized():
        print("CUDA is not initialized; Python GPU references cleared.")
        return
    try:
        torch.cuda.empty_cache()
        print(f"GPU cleanup: allocated {torch.cuda.memory_allocated()/1024**3:.2f} GiB; "
              f"reserved {torch.cuda.memory_reserved()/1024**3:.2f} GiB.")
    except RuntimeError as error:
        print(f"CUDA cleanup failed: {error}. Restart the kernel before loading another model.")

## 5. Tokenization and small model helpers

Keep the complete JSON target. Mask prompt/padding tokens with `-100` so training
loss applies only to the answer. Prediction decodes only newly generated tokens.
`predict()` is used for evaluation; Trainer consumes cached labels directly.
The progress bar reports examples/ETA, and Trainer supplies its own step/loss bar.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def prompt_tokens(tokenizer, record):
    return tokenizer.apply_chat_template([{"role": "user", "content": user_text(record)}],
        tokenize=True, add_generation_prompt=True, return_dict=False)

def tokenize_record(record, tokenizer):
    messages = [{"role": "user", "content": user_text(record)}]
    prompt = prompt_tokens(tokenizer, record)
    full = tokenizer.apply_chat_template(messages + [{"role": "assistant", "content":
        json.dumps(record["target"], separators=(",", ":"))}], tokenize=True, return_dict=False)
    if full[:len(prompt)] != prompt or len(full) > MAX_LENGTH:
        raise ValueError("Prompt boundary or complete target exceeds the token budget")
    return {"input_ids": full, "attention_mask": [1] * len(full),
            "labels": [-100] * len(prompt) + full[len(prompt):]}

train_dataset = Dataset.from_list([tokenize_record(r, tokenizer) for r in dataset.records("train")])
print(train_dataset)

def load_base():
    if not torch.cuda.is_available():
        raise RuntimeError("The model stages require a CUDA GPU")
    free, _ = torch.cuda.mem_get_info()
    print(f"Loading quantized Mistral; {free/1024**3:.2f} GiB GPU memory free.", flush=True)
    set_seed(SEED)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=config,
                                               device_map={"": 0}, attn_implementation="sdpa")

def release_gpu():
    gc.collect()
    if torch.cuda.is_initialized():
        torch.cuda.empty_cache()

def predict(model, records, label="Evaluation"):
    model.eval()
    model.config.use_cache = True
    predictions = []
    for record in tqdm(records, desc=label, unit="example"):
        ids = prompt_tokens(tokenizer, record)
        if len(ids) + MAX_NEW_TOKENS > MAX_LENGTH:
            raise ValueError("Complete prompt exceeds the generation budget")
        inputs = torch.tensor([ids], device=model.get_input_embeddings().weight.device)
        with torch.inference_mode():
            output = model.generate(input_ids=inputs, attention_mask=torch.ones_like(inputs),
                max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        predictions.append(tokenizer.decode(output[0, len(ids):], skip_special_tokens=True).strip())
    return predictions

def evaluate(model, split, label):
    predictions = predict(model, dataset.records(split), f"{label} / {split}")
    report = dataset.evaluate(predictions, split)
    save_json(RESULTS / f"{label}_{split}_predictions.json", predictions)
    save_json(RESULTS / f"{label}_{split}_metrics.json", report)
    return report


## 6. Five fixed examples before SFT

Freeze five validation cases, including positive repairs and an unrecoverable
window, before generation. Display the complete task/input, model answer, and
expected JSON separately. Reuse saved baseline responses when rerunning this run.
This cell performs only five generations; no full-set baseline delays training.


In [ ]:
demo_records = dataset.examples(count=5)
before_path = RESULTS / "before.json"
if before_path.exists():
    frozen = json.loads(before_path.read_text())
    assert frozen["ids"] == [r["id"] for r in demo_records]
    assert frozen["prompts"] == [user_text(r) for r in demo_records]
    before_predictions = frozen["predictions"]
else:
    model = load_base()
    try:
        before_predictions = predict(model, demo_records, "Before SFT: five examples")
        save_json(before_path, {"ids": [r["id"] for r in demo_records],
            "prompts": [user_text(r) for r in demo_records], "predictions": before_predictions})
    finally:
        del model
        release_gpu()
dataset.display_predictions(demo_records, before_predictions)


## 7. Starter-style LoRA training

The following function keeps the starter's LoRA configuration, TrainingArguments,
Trainer, and adapter save visible. One epoch on all 10,000 examples gives 625
optimizer steps. Prompt loss is masked; training never calls `generate()`.
Checkpoints are saved every 100 steps. Matching completed ranks are reused;
interrupted training resumes from the latest checkpoint.


In [ ]:
def train_adapter(rank):
    destination = RESULTS / f"rank_{rank}"
    adapter = destination / "adapter"
    rank_settings = {**settings, "rank": rank, "lora_alpha": 32, "lora_dropout": 0.05,
                     "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}
    config_path = destination / "settings.json"
    if config_path.exists() and json.loads(config_path.read_text()) != rank_settings:
        raise ValueError("Rank settings changed; choose a new RESULTS directory")
    if (destination / "training.json").exists() and (adapter / "adapter_model.safetensors").exists():
        print("Reusing saved adapter:", adapter)
        return adapter
    destination.mkdir(parents=True, exist_ok=True)
    save_json(config_path, rank_settings)
    checkpoint = get_last_checkpoint(str(destination))
    model = trainer = None
    try:
        model = prepare_model_for_kbit_training(load_base())
        model = get_peft_model(model, LoraConfig(r=rank, lora_alpha=32, lora_dropout=0.05,
            target_modules=rank_settings["target_modules"], bias="none", task_type="CAUSAL_LM"))
        model.config.use_cache = False
        model.print_trainable_parameters()
        arguments = TrainingArguments(output_dir=str(destination),
            per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=ACCUMULATION,
            num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
            gradient_checkpointing=True, logging_steps=5, logging_first_step=True, disable_tqdm=False,
            save_steps=100, save_total_limit=2, report_to="none", seed=SEED, data_seed=SEED,
            remove_unused_columns=False)
        trainer = Trainer(model=model, args=arguments, train_dataset=train_dataset,
            processing_class=tokenizer, data_collator=DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, label_pad_token_id=-100))
        torch.cuda.reset_peak_memory_stats()
        started = time.monotonic()
        result = trainer.train(resume_from_checkpoint=checkpoint)
        if not math.isfinite(result.training_loss):
            raise RuntimeError("Training loss is not finite")
        trainer.model.save_pretrained(adapter)
        tokenizer.save_pretrained(adapter)
        trainer.state.save_to_json(str(destination / "trainer_state.json"))
        save_json(destination / "training.json", {"rank": rank, "examples": len(train_dataset),
            "steps": trainer.state.global_step, "epochs": trainer.state.epoch,
            "loss": result.training_loss, "seconds": time.monotonic() - started,
            "peak_allocated_gib": torch.cuda.max_memory_allocated()/1024**3,
            "peak_reserved_gib": torch.cuda.max_memory_reserved()/1024**3})
        return adapter
    finally:
        del trainer, model
        release_gpu()


## 8. Train the primary adapter

This cell trains directly from the cached tokenized input/JSON-label pairs.
No model responses or recovery tools are used to create labels here.


In [ ]:
assert before_path.exists(), "Run the five-example baseline first"
selected_adapter = train_adapter(LORA_RANK)
print("Saved primary adapter:", selected_adapter)


## 9. The same five examples after SFT

Reload the saved adapter onto a fresh base and use the same five IDs, prompts,
and generation settings. This small comparison illustrates behavior; the full
held-out results below support broader conclusions. After a kernel restart,
rerun setup through Section 5, Section 6 (it reuses the baseline), and the
Section 7 function definition. You can then run this cell without retraining.


In [ ]:
frozen = json.loads((RESULTS / "before.json").read_text())
demo_records = dataset.examples(count=5)
assert frozen["ids"] == [r["id"] for r in demo_records]
assert frozen["prompts"] == [user_text(r) for r in demo_records]
selected_adapter = RESULTS / f"rank_{LORA_RANK}" / "adapter"
model = PeftModel.from_pretrained(load_base(), selected_adapter)
try:
    after_predictions = predict(model, demo_records, "After SFT: five examples")
    dataset.display_predictions(demo_records, after_predictions, frozen["predictions"])
    save_json(RESULTS / "after.json", {**frozen, "predictions": after_predictions})
    save_json(RESULTS / "after_metrics.json", score_predictions(after_predictions, demo_records))
finally:
    del model
    release_gpu()


## 10. Rank comparison on validation data

An explicit long job: train/reuse ranks 8/16/32, then generate validation answers
for each. This occurs after primary SFT and does not generate training labels.
It reports rank differences without changing the configured primary adapter.

The assignment requires exploring LoRA rank values and documenting their
performance impact. The values 8/16/32 compare a smaller adapter, the primary
rank 16, and a larger adapter. These are separate training runs, each starting
from the original base; a completed matching rank-16 adapter is reused.


In [ ]:
if RUN_RANK_STUDY:
    rank_reports = []
    for rank in RANKS:
        adapter = train_adapter(rank)
        model = PeftModel.from_pretrained(load_base(), adapter)
        try:
            rank_reports.append({"rank": rank, **evaluate(model, "validation", f"rank_{rank}")})
        finally:
            del model
            release_gpu()
    save_json(RESULTS / "rank_comparison.json", rank_reports)
else:
    print("Rank study disabled; rerun settings after editing its switch.")


## 11. Full held-out comparison

Another explicit long job: generate 2,000 test answers with the original base and
2,000 with the trained primary adapter. Score strict JSON, recoverability,
operation order, complete answers, and category/composition breakdowns. Routine
evaluation never executes recovery tools. The unadapted base remains the original
model condition even though its full measurement occurs after training in cell order.


In [ ]:
if RUN_FULL_TEST_COMPARISON:
    for label in ("base", "selected"):
        model = load_base()
        if label == "selected":
            model = PeftModel.from_pretrained(model, RESULTS / f"rank_{LORA_RANK}" / "adapter")
        try:
            evaluate(model, "test", label)
        finally:
            del model
            release_gpu()
else:
    print("Full test comparison disabled; rerun settings after editing its switch.")


## 12. Results and optional final tool execution

Display metrics from saved model predictions. The optional final demonstration
executes the model's predicted repair operations and checks reconstruction; it
does not belong to training or routine scoring. Nonrecoverable windows are
omitted completely. Synthetic corruption and five examples alone do not establish
performance on natural errors. GPU experiments and outputs stay outside Git/PyPI.

Sources: source CSV in [NLP-proj5](https://github.com/Absolute-Convergence/NLP-proj5),
the supplied local starter, [PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization),
[Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), and
[chat templates](https://huggingface.co/docs/transformers/chat_templating).
AI/source assistance is summarized in the separate local `proj5doc.md`.


In [ ]:
metrics = ["recoverability_accuracy", "operation_sequence_accuracy", "full_answer_accuracy", "format_validity"]
reports = {name: json.loads((RESULTS / f"{name}_test_metrics.json").read_text())
           for name in ("base", "selected") if (RESULTS / f"{name}_test_metrics.json").exists()}
if len(reports) == 2:
    display(pd.DataFrame({name: {m: r[m] for m in metrics} for name, r in reports.items()}))
    display(pd.DataFrame(reports["selected"]["confusion_matrix"]).T)
    display(pd.DataFrame(reports["selected"]["breakdowns"]).T[metrics])
if (RESULTS / "rank_comparison.json").exists():
    display(pd.DataFrame(json.loads((RESULTS / "rank_comparison.json").read_text())).set_index("rank")[metrics])
if RUN_FINAL_DEMO:
    predictions = json.loads((RESULTS / "selected_test_predictions.json").read_text())
    display(dataset.execute_predictions(predictions, RESULTS / "final_execution"))
